<a href="https://colab.research.google.com/github/postnicov/ResazurinResorufin/blob/main/Munsell_to_RGB_Lab_Image.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Munsell codes → RGB, CIE Lab, and DOCX export with colour rectangles

This notebook unifies two workflows:

1. **Reference conversion**: Read `MilkMunsellCodes.csv` directly from GitHub (raw URL) to compute RGB and CIE L*a*b* from Munsell codes, then export `MilkMunsell_RGB_Lab.csv`.
2. **Manual upload**: Upload a custom CSV of Munsell codes (same column structure) and run the same conversion.

For both, it also:

- Generates rectangular colour swatch images (via Pillow).
- Builds DOCX tables with the colour swatches inserted in the last column.
- Includes explicit RGB (0–255) columns `R`, `G`, `B` in the DOCX tables.
- Saves DOCX files that you can download from Colab’s file browser or via `files.download(...)`.

In [19]:
#@title Imports and dependency check
import sys
import subprocess

def ensure_package(pkg_name, pip_name=None):
    """Import a package, installing it via pip if missing."""
    if pip_name is None:
        pip_name = pkg_name
    try:
        __import__(pkg_name)
        print(f"Package {pkg_name} already available.")
    except ImportError:
        print(f"Package {pkg_name} not found, installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
        __import__(pkg_name)
        print(f"Package {pkg_name} installed and imported.")

# Ensure required packages
ensure_package("colour", "colour-science")
ensure_package("docx", "python-docx")
ensure_package("PIL", "pillow")
ensure_package("pandas", "pandas")
ensure_package("numpy", "numpy")

# Now import them normally
import colour
import numpy as np
import pandas as pd
from PIL import Image
from docx import Document
from docx.shared import Inches
import os

# For Colab downloads (optional)
try:
    from google.colab import files
except ImportError:
    files = None

os.makedirs("output", exist_ok=True)
illuminant_C = colour.CCS_ILLUMINANTS["CIE 1931 2 Degree Standard Observer"]["C"]
illuminant_D65 = colour.CCS_ILLUMINANTS["CIE 1931 2 Degree Standard Observer"]["D65"]

Package colour already available.
Package docx already available.
Package PIL already available.
Package pandas already available.
Package numpy already available.


In [20]:
#@title Conversion functions
def munsell_to_rgb_lab(code: str):
    """Convert a Munsell code string to XYZ, Lab (Illuminant C), sRGB, and RGB255."""
    xyY = colour.notation.munsell_colour_to_xyY(code)
    XYZ = colour.xyY_to_XYZ(xyY)
    Lab = colour.XYZ_to_Lab(XYZ, illuminant=illuminant_D65)
    RGB = np.clip(colour.XYZ_to_sRGB(XYZ), 0, 1)
    RGB255 = np.round(RGB * 255).astype(int)
    return XYZ, Lab, RGB, RGB255

def convert_munsell_dataframe(df_codes: pd.DataFrame) -> pd.DataFrame:
    """
    Given a DataFrame with columns ['Item', 'Munsell code'],
    return full RGB/Lab table.
    """
    rows = []
    for _, row in df_codes.iterrows():
        code = str(row["Munsell code"]).strip()
        XYZ, Lab, RGB, RGB255 = munsell_to_rgb_lab(code)
        rows.append({
            "Item": row["Item"],
            "Munsell code": code,
            "L*": Lab[0],
            "a*": Lab[1],
            "b*": Lab[2],
            "R": RGB255[0],
            "G": RGB255[1],
            "B": RGB255[2],
            "R_norm": RGB[0],
            "G_norm": RGB[1],
            "B_norm": RGB[2],
        })
    return pd.DataFrame(rows)

In [21]:
#@title DOCX generation with RGB columns
def add_colour_table_docx(df: pd.DataFrame, docx_path: str, title: str):
    """
    Create swatch images and a DOCX table with rectangles in the last column.
    Columns: Item, Munsell code, L*, a*, b*, R, G, B, Colour sample.
    """
    # Generate swatch images
    swatch_paths = []
    base_name = os.path.splitext(os.path.basename(docx_path))[0]
    for i, r in df.iterrows():
        rgb = (int(r["R"]), int(r["G"]), int(r["B"]))
        img = Image.new("RGB", (200, 80), rgb)
        path = os.path.join("output", f"swatch_{base_name}_{i+1}.png")
        img.save(path)
        swatch_paths.append(path)

    # Build DOCX document
    doc = Document()
    doc.add_heading(title, level=1)

    cols = 9  # Item, Munsell, L*, a*, b*, R, G, B, Colour sample
    table = doc.add_table(rows=1, cols=cols)
    hdr = table.rows[0].cells
    hdr[0].text = "Item"
    hdr[1].text = "Munsell code"
    hdr[2].text = "L*"
    hdr[3].text = "a*"
    hdr[4].text = "b*"
    hdr[5].text = "R"
    hdr[6].text = "G"
    hdr[7].text = "B"
    hdr[8].text = "Colour sample"

    for i, r in df.iterrows():
        cells = table.add_row().cells
        cells[0].text = str(r["Item"])
        cells[1].text = str(r["Munsell code"])
        cells[2].text = f"{r['L*']:.3f}"
        cells[3].text = f"{r['a*']:.3f}"
        cells[4].text = f"{r['b*']:.3f}"
        cells[5].text = str(int(r["R"]))
        cells[6].text = str(int(r["G"]))
        cells[7].text = str(int(r["B"]))
        run = cells[8].paragraphs[0].add_run()
        run.add_picture(swatch_paths[i], width=Inches(1.2), height=Inches(0.5))

    doc.save(docx_path)
    print(f"DOCX saved to: {docx_path}")
    if files is not None:
        files.download(docx_path)

In [22]:
#@title  Reference conversion from GitHub
# Read MilkMunsellCodes.csv directly from GitHub raw URL
url = "https://raw.githubusercontent.com/postnicov/ResazurinResorufin/refs/heads/main/Data/MilkMunsellCodes.csv"
df_codes = pd.read_csv(url)

# Ensure expected column names; adjust if GitHub file differs
df_codes.columns = ["Item", "Munsell code"]
df_codes["Munsell code"] = df_codes["Munsell code"].astype(str).str.strip()

df_ref = convert_munsell_dataframe(df_codes)
df_ref.to_csv("MilkMunsell_RGB_Lab.csv", index=False)
df_ref

,Item,Munsell code,L*,a*,b*,R,G,B,R_norm,G_norm,B_norm
0,L6,5PB 7/4,70.860848,3.910739,-18.896755,162,173,208,0.636710,0.676890,0.813866
1,L4,10PB 7/5.5,70.860848,13.160033,-24.176239,176,167,217,0.688514,0.656429,0.852244
2,L3,5P 7/4,70.860848,14.214762,-15.726805,186,166,202,0.730803,0.650568,0.792579
3,L2,10P 7/8,70.860848,33.248978,-18.373021,218,152,207,0.854949,0.597248,0.813490
4,LP6,10B 9/2,90.181318,1.321476,-10.743478,219,227,247,0.859508,0.889792,0.969538
5,LP5,2.5PB 9/2,90.181318,3.316808,-10.805471,223,226,247,0.875808,0.885009,0.970244
6,LP4,7.5PB 9/2,90.181318,5.929343,-10.563358,229,224,247,0.898026,0.878522,0.968772
7,LP3,5P 9/2,90.181318,8.172780,-9.405613,235,222,245,0.920267,0.872527,0.960467
8,LP2,2.5RP 9/2,90.181318,9.856067,-6.242772,241,221,239,0.944654,0.867211,0.937240
9,LP1,5RP 9/3,90.181318,14.908461,-5.316046,251,218,237,0.985773,0.853682,0.931058


In [23]:
#@title DOCX export for reference
docx_ref_path = os.path.join("output", "MilkColours_reference.docx")
add_colour_table_docx(df_ref, docx_ref_path, "Milk sample colours (reference conversion, v3, no hex)")

DOCX saved to: output/MilkColours_reference.docx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#@title To process own data: manual upload of csv with ID and Munsell code
# In Colab, run this cell, then upload your CSV when prompted.
if files is not None:
    uploaded = files.upload()
    fname = next(iter(uploaded.keys()))
else:
    fname = "YourManualMunsellCodes.csv"  # adjust if running locally

df_manual_codes = pd.read_csv(fname)
# Expect columns ['Item', 'Munsell code']
df_manual_codes["Munsell code"] = df_manual_codes["Munsell code"].astype(str).str.strip()

df_manual = convert_munsell_dataframe(df_manual_codes)
out_csv_manual = "Manual_Munsell_RGB_Lab_v3_nohex.csv"
df_manual.to_csv(out_csv_manual, index=False)
df_manual

In [ ]:
#@title DOCX export for manual upload
docx_manual_path = os.path.join("output", "MilkColours_manual_v3_nohex.docx")
add_colour_table_docx(df_manual, docx_manual_path, "Milk sample colours (manual upload conversion, v3, no hex)")